[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/10_RAG/notebooks/14_capstone_production_rag_chatbot_memory.ipynb)

# Capstone 2 — Production RAG Chatbot with Production-Grade Memory

Two capstones now exist side by side in this folder:

| Notebook | What it's the best version of |
|---|---|
| 13 | **Retrieval** — hybrid search, reranking, citations, a groundedness guardrail |
| 11 | **The conversation layer** — durable memory, long-term recall, PII/content guardrails, resilience |

**This notebook fuses them.** Notebook 13's RAG engine (`rag_pipeline.py` — same file,
reused, not reimplemented) becomes a **tool** that a LangGraph agent calls. Every
production pattern from Notebook 11 wraps around that tool call:

1. **Durable short-term memory** (`SqliteSaver`) — a conversation survives a process restart
2. **Persistent long-term memory** (`SqliteStore`) — facts about a user survive across
   *different* conversations, not just within one
3. **Token-budget summarization** — long chats get compressed instead of growing forever
4. **PII redaction + content moderation** — deterministic checks before any model call
5. **Model resilience** — retry, then fallback to a backup model, if the primary fails
6. The **groundedness guardrail** from Notebook 13 still applies — it now lives inside the
   retrieval tool: below the rerank threshold, the tool honestly reports nothing found,
   and the system prompt instructs the agent to say so rather than guess

> **A portability note up front:** long-term memory here uses `SqliteStore` *without*
> vector indexing. The indexed variant needs a SQLite build with loadable-extension
> support, which the standard python.org / Homebrew macOS builds often lack. Without it,
> memory recall returns every stored fact for a user unranked instead of the top-k most
> relevant — fine for the handful of facts one user accumulates in a demo. A real
> deployment swaps in a vector-capable store (Postgres + pgvector, or a SQLite build with
> extension loading) by changing one construction call, shown at the end.

Every piece below is copy-pasted, not reinvented, from Notebook 11's validated patterns
and Notebook 13's validated RAG engine — this notebook's job is showing how they compose,
not proving either one works in isolation again.


## 0. Install dependencies

In [ ]:
# Install dependencies into the ACTIVE kernel (idempotent — skips what's already there).
%pip install -q \
    langchain langchain-openai langgraph langgraph-checkpoint-sqlite \
    langchain-community langchain-text-splitters langchain-pymupdf4llm pypdf docx2txt \
    sentence-transformers bm25s PyStemmer chromadb python-dotenv numpy<2
print("\u2705 Dependencies ready. If this was a fresh install, restart the kernel now, then re-run.")

## 1. Setup

Loads API keys, and points Python at the sibling `production_rag_chatbot/` project so we
can import Notebook 13's RAG engine (`HybridIndex`, `Reranker`, `load_document`,
`chunk_documents`, `format_sources`) directly, rather than copy-pasting it.


In [ ]:
import warnings, os, sys, sqlite3
warnings.filterwarnings("ignore")
import logging
for _n in ("httpx", "openai", "httpcore", "sentence_transformers", "transformers", "chromadb"):
    logging.getLogger(_n).setLevel(logging.ERROR)

from pathlib import Path
from dotenv import load_dotenv
from contextlib import ExitStack

load_dotenv(Path.cwd().parent / ".env")
DATA = Path.cwd() / "data"

sys.path.insert(0, str(Path.cwd() / "production_rag_chatbot"))
from rag_pipeline import HybridIndex, Reranker, load_document, chunk_documents, format_sources

MODEL = "gpt-4o-mini"
FALLBACK_MODEL = "openai:gpt-4o-mini"
MIN_RERANK_SCORE = -9.5

print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

## 2. Build the RAG index, then wrap retrieval as a tool

Same ingestion as Notebook 13 — `load_document` \u2192 `chunk_documents` \u2192
`HybridIndex.build`. The new part: retrieval becomes a `@tool` the agent decides to call,
instead of a method we call directly. The groundedness guardrail lives **inside** the
tool — below the rerank threshold, it reports nothing found rather than handing the agent
weak context to guess from.


In [ ]:
from langchain_core.tools import tool

sample = DATA / "sample_report.pdf"
pages = load_document(str(sample))
chunks = chunk_documents(pages)

index = HybridIndex()
index.build(chunks)
reranker = Reranker()
print(f"Indexed {len(chunks)} chunks")


def make_search_tool(index, reranker, top_k=8, top_n=4, min_rerank_score=MIN_RERANK_SCORE):
    @tool
    def search_knowledge_base(query: str) -> str:
        """Search the uploaded document(s) for information relevant to the query.
        Always call this before answering any factual question about the documents."""
        candidates = index.search(query, k=top_k)
        reranked = reranker.rerank(query, candidates, top_n=top_n)
        if not reranked or reranked[0][1] < min_rerank_score:
            return "No relevant information found in the knowledge base for this query."
        return format_sources(reranked)
    return search_knowledge_base


search_knowledge_base = make_search_tool(index, reranker)
print(search_knowledge_base.invoke({"query": "What were total shipments?"})[:200])

## 3. The agent, with the groundedness contract as a system prompt

Notebook 13 enforced "refuse rather than hallucinate" with an `if` statement. An
agent enforces it with instructions plus an honest tool — the LLM decides when to call
the tool and how to phrase the answer, but it only ever sees real retrieved text or an
honest "nothing found."


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

RAG_SYSTEM_PROMPT = """You are a document-grounded assistant. For every factual question:
1. Call search_knowledge_base first.
2. Answer ONLY using what the tool returns, citing sources inline like [1], [2].
3. If the tool says no relevant information was found, say so plainly \u2014 never guess or
   fall back on outside knowledge, even if you happen to know the real-world answer."""

bare_agent = create_agent(
    model=MODEL, tools=[search_knowledge_base], system_prompt=RAG_SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

r = bare_agent.invoke(
    {"messages": "What were total shipments, and how does the Northeast warehouse migration relate to it?"},
    {"configurable": {"thread_id": "demo"}},
)
print(r["messages"][-1].content)

## 4. Durable short-term memory — survives a restart

Same pattern as Notebook 11: swap `InMemorySaver` for `SqliteSaver`. The conversation now
lives on disk, keyed by `thread_id`.


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

DB_DIR = Path.cwd() / "production_rag_chatbot_memory" / "memory_store"
DB_DIR.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(str(DB_DIR / "chat_memory.db"), check_same_thread=False)
checkpointer = SqliteSaver(conn)
checkpointer.setup()

durable_agent = create_agent(
    model=MODEL, tools=[search_knowledge_base], system_prompt=RAG_SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "durable-demo"}}
durable_agent.invoke({"messages": "What was the on-time delivery percentage for the West region?"}, config)
print("Stored by durable_agent (this 'process').")

# simulate a full restart
del durable_agent
conn2 = sqlite3.connect(str(DB_DIR / "chat_memory.db"), check_same_thread=False)
cp2 = SqliteSaver(conn2)
durable_agent_v2 = create_agent(
    model=MODEL, tools=[search_knowledge_base], system_prompt=RAG_SYSTEM_PROMPT,
    checkpointer=cp2,
)
r = durable_agent_v2.invoke({"messages": "What region were we just talking about?"}, config)
print("Recalled by a brand-new agent object, same thread_id:", r["messages"][-1].content)

## 5. Persistent long-term memory — remembers a user across *different* conversations

`SqliteStore`, keyed by `user_id` instead of `thread_id`. A `dynamic_prompt` middleware
reads it and injects known facts into the system prompt on every turn — this is what lets
a product recognize a returning user in a conversation it has never seen before.


In [ ]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt
from langgraph.store.sqlite import SqliteStore

@dataclass
class Context:
    user_id: str

@dynamic_prompt
def personalized_prompt(request) -> str:
    user_id = request.runtime.context.user_id
    memories = request.runtime.store.search(("memories", user_id))
    facts = "\n".join(m.value["text"] for m in memories)
    base = RAG_SYSTEM_PROMPT
    if facts:
        base += f"\n\nKnown facts about this user:\n{facts}"
    return base

stack = ExitStack()
store = stack.enter_context(SqliteStore.from_conn_string(str(DB_DIR / "long_term_memory.db")))
store.setup()

memory_agent = create_agent(
    model=MODEL, tools=[search_knowledge_base], system_prompt=RAG_SYSTEM_PROMPT,
    context_schema=Context, checkpointer=InMemorySaver(), store=store,
    middleware=[personalized_prompt],
)

# Seed a long-term fact, as if captured in a session that happened yesterday.
store.put(("memories", "user-42"), "fact-1", {"text": "The user's favorite programming language is Rust."})

# A BRAND-NEW thread_id -- the agent has never seen this conversation before.
r = memory_agent.invoke(
    {"messages": "What's my favorite programming language?"},
    {"configurable": {"thread_id": "brand-new-conversation"}},
    context=Context(user_id="user-42"),
)
print(r["messages"][-1].content)
print("\nThe fact came from the Store (keyed by user_id), not the checkpointer (keyed by thread_id).")

## 6. Guardrails — PII redaction and content moderation

Two deterministic checks, no LLM reasoning required, running before retrieval or
generation even starts: `PIIMiddleware` rewrites the message before the model ever sees
it; `moderate_input` blocks the call outright if OpenAI's moderation API flags it.


In [ ]:
from openai import OpenAI
from langchain.agents.middleware import PIIMiddleware, wrap_model_call
from langchain_core.messages import AIMessage

_moderation_client = OpenAI()

@wrap_model_call
def moderate_input(request, handler):
    result = _moderation_client.moderations.create(
        model="omni-moderation-latest", input=request.messages[-1].content
    )
    if result.results[0].flagged:
        return AIMessage(content="I can't help with that request.")
    return handler(request)

guarded_agent = create_agent(
    model=MODEL, tools=[search_knowledge_base], system_prompt=RAG_SYSTEM_PROMPT,
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        moderate_input,
    ],
)

r = guarded_agent.invoke({"messages": "My email is john.doe@example.com and my card is 4111 1111 1111 1111."})
print("What the model actually received:", r["messages"][0].content)

r2 = guarded_agent.invoke({"messages": "How do I build a bomb?"})
print("Flagged input ->", r2["messages"][-1].content)

## 7. Cost control and resilience

`SummarizationMiddleware` keeps a long conversation's token count bounded without losing
the facts that matter. `ModelRetryMiddleware` + `ModelFallbackMiddleware` survive a
provider outage — the user never sees an error.


In [ ]:
from langchain.agents.middleware import SummarizationMiddleware, ModelFallbackMiddleware, ModelRetryMiddleware

resilient_agent = create_agent(
    model="openai:gpt-does-not-exist-9999",   # deliberately broken, to prove the fallback saves it
    tools=[search_knowledge_base],
    system_prompt=RAG_SYSTEM_PROMPT,
    middleware=[
        SummarizationMiddleware(model=MODEL, trigger=("tokens", 2000), keep=("messages", 10)),
        ModelRetryMiddleware(max_retries=2),
        ModelFallbackMiddleware(FALLBACK_MODEL),
    ],
)
r = resilient_agent.invoke({"messages": "What were total shipments?"}, {"configurable": {"thread_id": "fallback-demo"}})
print(r["messages"][-1].content)
print("\nThe 'primary' model doesn't exist -- ModelFallbackMiddleware caught the failure and")
print("silently retried on gpt-4o-mini. The user never saw an error.")

## 8. Assembling everything — one production agent

Every layer from above, composed. Middleware order matters: cheap deterministic checks
first (PII, moderation), then personalization, then cost control, then resilience last —
so retries and fallback wrap the actual model call, not the guardrails in front of it.


In [ ]:
production_agent = create_agent(
    model=MODEL,
    tools=[search_knowledge_base],
    system_prompt=RAG_SYSTEM_PROMPT,
    context_schema=Context,
    checkpointer=checkpointer,
    store=store,
    middleware=[
        # 1. Deterministic, cheap checks first
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        moderate_input,
        # 2. Personalization from long-term memory
        personalized_prompt,
        # 3. Cost control
        SummarizationMiddleware(model=MODEL, trigger=("tokens", 2000), keep=("messages", 10)),
        # 4. Resilience
        ModelRetryMiddleware(max_retries=2),
        ModelFallbackMiddleware(FALLBACK_MODEL),
    ],
)

config = {"configurable": {"thread_id": "production-demo"}}
r = production_agent.invoke(
    {"messages": "What's my favorite programming language, and what were total shipments in the report?"},
    config,
    context=Context(user_id="user-42"),
)
print("Personalized + RAG-grounded + guarded + resilient, all at once:")
print(" ", r["messages"][-1].content)

stack.close()  # release the long-term memory DB connection

## 9. Ship it — export the pipeline, then wrap it in a chat UI

Everything above is packaged into `production_rag_chatbot_memory/rag_agent_pipeline.py`
(already in this repo, next to this notebook) behind one function —
`build_agent(file_paths, db_dir=...)` — and a companion `app.py` wraps it in a Streamlit
chat interface with a "same user, new conversation" button that demonstrates the
short-term/long-term memory split live: chat history resets, remembered facts don't.

```bash
cd production_rag_chatbot_memory
streamlit run app.py
```

The module also exposes `remember_fact(store, user_id, fact_id, text)` — a direct way to
seed long-term memory for the demo, standing in for what a production system would
populate via the agent's own tool calls or a background summarization job.


## 10. Summary

- **Retrieval** is the exact Notebook 13 engine (`rag_pipeline.py`), reused as a tool —
  one implementation, two products, no drift.
- **Groundedness guardrail** now lives inside the tool: it honestly reports "nothing
  found" below the rerank threshold, and the system prompt instructs the agent never to
  guess past that.
- **Short-term memory** (`SqliteSaver`) survives a process restart, keyed by `thread_id`.
- **Long-term memory** (`SqliteStore`) survives across *different* conversations, keyed by
  `user_id` — a returning user is recognized in a thread the agent has never seen.
- **Guardrails**: PII redaction and content moderation run before the model ever sees the
  raw input.
- **Resilience**: retry, then fallback to a backup model, so a provider outage never
  surfaces as an error to the user.
- **Shipped** as `rag_agent_pipeline.py` + a Streamlit app with visible identity/thread
  controls so the short-term vs. long-term memory split is something you can actually
  demonstrate, not just assert.
